In [1]:
!pip install -U seaborn
!pip install -U scikit-learn
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Purpose of Notebook

The purpose is to measure the cluster quality of each clustering for hyperparameter tuning

In [4]:
from pathlib import Path

DATA_FOLDER = Path('/content/drive/MyDrive/DSCI599/data')
VIZ_FOLDER = Path('/content/drive/MyDrive/DSCI599/viz')

assert DATA_FOLDER.exists()
assert VIZ_FOLDER.exists()


In [5]:
import pandas as pd
# currently using one
df = pd.read_csv(DATA_FOLDER / "df_cluster.csv")
df.head()

,int_source_ip,destination_port,protocol,num_scanned_destination_ips,num_unique_flows,num_packets,same_packet_size_flag,avg_packet_size,num_unique_Bs_scanned,num_unique_Cs_scanned,num_unique_Ds_scanned,num_scanned_24_blocks,num_non_conficker_destinations,time_activity,ip_scanned_rate,graph_labels,normal_labels
0,16968188,23,6,316.0,316.0,945.0,1.0,60.0,175.0,182.0,177.0,316.0,65.0,3540.0,0.089266,16,13
1,17372742,445,6,661.0,661.0,1322.0,1.0,48.0,125.0,234.0,126.0,654.0,661.0,3540.0,0.186723,15,-1
2,17384594,445,6,169.0,169.0,329.0,1.0,48.0,98.0,117.0,89.0,169.0,169.0,3360.0,0.050298,-1,-1
3,17400985,23,6,256.0,256.0,762.0,1.0,60.0,1.0,1.0,256.0,1.0,128.0,0.0,-1.000000,16,14
4,17401572,445,6,571.0,571.0,1142.0,1.0,48.0,124.0,232.0,125.0,564.0,571.0,3540.0,0.161299,15,-1


In [6]:
df['graph_labels'] = pd.Categorical(df['graph_labels'])
df['normal_labels'] = pd.Categorical(df['normal_labels'])

In [7]:
features = df.drop(columns=[
    "int_source_ip",
    "destination_port",
    "protocol",
    "graph_labels",
    "normal_labels"
])

features.head()

,num_scanned_destination_ips,num_unique_flows,num_packets,same_packet_size_flag,avg_packet_size,num_unique_Bs_scanned,num_unique_Cs_scanned,num_unique_Ds_scanned,num_scanned_24_blocks,num_non_conficker_destinations,time_activity,ip_scanned_rate
0,316.0,316.0,945.0,1.0,60.0,175.0,182.0,177.0,316.0,65.0,3540.0,0.089266
1,661.0,661.0,1322.0,1.0,48.0,125.0,234.0,126.0,654.0,661.0,3540.0,0.186723
2,169.0,169.0,329.0,1.0,48.0,98.0,117.0,89.0,169.0,169.0,3360.0,0.050298
3,256.0,256.0,762.0,1.0,60.0,1.0,1.0,256.0,1.0,128.0,0.0,-1.000000
4,571.0,571.0,1142.0,1.0,48.0,124.0,232.0,125.0,564.0,571.0,3540.0,0.161299


In [ ]:
import numpy as np
numeric_columns = features.select_dtypes(include=[np.number]).columns
numeric_columns

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device)
src_embeddings = torch.load(DATA_FOLDER / "source_ip_embeddings.pt", map_location=device)
src_embeddings.requires_grad = False
src_embeddings.shape

# Evaluation
Let's evaluate using sillhouette score.

Sillhouette score:
$(b - a) / max(a, b)$

where $a$ is mean intra-cluster distance. $b$ is inter-cluster distance.

Basically, the higher the score, the more better the cluster distance.

In [ ]:
from sklearn.metrics import silhouette_score

np_feat = features.to_numpy()
np_embeddings = src_embeddings.cpu().numpy()

In [ ]:
raw_labels = df['normal_labels'].to_numpy()
graph_labels = df['graph_labels'].to_numpy()

In [ ]:
raw_score = silhouette_score(np_feat, raw_labels)

In [ ]:
graph_score = silhouette_score(np_embeddings, graph_labels)